# Preprocessing

Pada tahap ini akan dilakukan perbaikan terhadap baris suspensi yang ditemukan pada EDA, menyesuaikan harga terhadap stock split, menghitung indikator teknikal, dan membentuk fitur target.

In [12]:
# Import Library
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sys
from pathlib import Path

# Path Folder Root
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

# Import Fungsi
from feature_pipeline import CALENDAR_COLUMNS, compute_technical_indicators, repair_suspension_rows
from pipeline_utils import adjust_prices_for_splits, attach_calendar_targets, show_table
from notebook_config import configure_notebook, feature_columns as features, target, tickers

In [ ]:
# Konfigurasi Path Folder
root_dir, _ = configure_notebook()
interim_dir = root_dir / "data" / "interim"
processed_dir = root_dir / "data" / "processed"

# Inisialisasi Kolom Kunci dan Kolom Harga
key_columns = ["Kode Saham", "snapshot_date"]
price_columns = ["Open Price", "Tertinggi", "Terendah", "Penutupan"]

## Konfigurasi Grafik
plt.rcParams["figure.dpi"] = 120 # DPI 
plt.rcParams["font.family"] = "DejaVu Sans" # Font

# Konfigurasi nama tampilan kolom
display_names = {
    "snapshot_date": "Tanggal",
    "target_date": "Tanggal Target",
    "Kode Saham": "Ticker",
    "close_adj": "Adjusted Close",
    "simple_return": "Simple Return",
    "target_next_return": "Target Return",
    "SMA10": "SMA 10",
    "SMA20": "SMA 20",
    "EMA10": "EMA 10",
    "RSI14": "RSI 14",
    "Signal": "MACD Signal",
    "BB_Upper": "Bollinger Upper",
    "BB_Lower": "Bollinger Lower",
    "Stochastic_%K": "Stochastic %K",
    "ATR14": "ATR 14"
}

## Load Data

In [18]:
# Load Data hasil notebook 01
raw_data = pd.read_parquet(interim_dir / "panel_raw.parquet")
# Ambil Kode Saham yang jadi objek perancangan aja yang ada di tickers
raw_df = raw_data.loc[raw_data["Kode Saham"].isin(tickers)].copy()
# Sort per saham dan tanggal lalu re-indeks
raw_df = raw_df.sort_values(key_columns, kind="stable").reset_index(drop=True)

## Fill Data

Tahap EDA sebelumnya menemukan OHLV bernilai nol pada emiten BNLI pada tanggal 2025-09-30. Data OHL pada hari tersebut diisi dengan Close hari sebelumnya, sedangkan Volume dipertahankan karena memang tidak terdapat perdagangan.

In [23]:
# Memperbaiki Baris Suspensi
repaired_df = repair_suspension_rows(raw_df)

# Cari baris yang harganya berubah karena diperbaiki
view_columns = key_columns + price_columns + ["Volume"]
repaired_rows = raw_df[price_columns].ne(repaired_df[price_columns]).any(axis=1)

# Display baris sebelum dan sesudah diperbaiki
print("Sebelum")
show_table(raw_df.loc[repaired_rows, view_columns], display_names=display_names)
print("Sesudah")
show_table(repaired_df.loc[repaired_rows, view_columns], display_names=display_names)

Sebelum


,Ticker,Tanggal,Open Price,Tertinggi,Terendah,Penutupan,Volume
12959,BNLI,2025-09-30,0,0,0,"5,950",0


Sesudah


,Ticker,Tanggal,Open Price,Tertinggi,Terendah,Penutupan,Volume
12959,BNLI,2025-09-30,"5,950","5,950","5,950","5,950",0


## Penyesuaian Harga terhadap Stock Split

Rasio perubahan Listed Shares yang berupa faktor bulat ditandai sebagai stock split. Harga sebelum tanggal split dikalikan faktor kebalikannya secara mundur, sedangkan volumenya akan dikalikan rasio saham.

In [27]:
# Penyesuaian Harga Stock Split
calendar_data, split_events = adjust_prices_for_splits(repaired_df)

show_table(
    split_events,
    display_names={
        "Kode Saham": "Ticker",
        "snapshot_date": "Tanggal split",
        "listed_shares_ratio": "Rasio Listed Shares",
    },
)

,Ticker,Tanggal split,Rasio Listed Shares
0,BBCA,2021-10-13,5
1,BBNI,2023-10-06,2
2,BMRI,2023-04-04,2


In [28]:
# Simpan Kalender Harga
calendar_data = calendar_data[CALENDAR_COLUMNS].copy()
calendar_data.to_parquet(processed_dir / "panel_calendar.parquet", index=False)

## Feature Engineering

Indikator dihitung per saham menggunakan kolom Tertinggi, Terendah, dan Penutupan yang telah disesuaikan terhadap stock split.

In [ ]:
# Hitung Indikator Teknikal per saham
processed_list = [
    compute_technical_indicators(group) for _, group in calendar_data.groupby("Kode Saham")
]

# Gabung semua dan sort
fe_df = pd.concat(processed_list, ignore_index=True)
fe_df = fe_df.sort_values(["Kode Saham", "snapshot_date"]).reset_index(drop=True)

## Membentuk Target Return

Fitur pada tanggal t dipasangkan dengan simple return pada hari bursa berikutnya. Baris dengan indikator atau target yang belum lengkap dikeluarkan dari data model.

In [ ]:
# Pasangkan fitur hari ini dengan return hari bursa berikutnya
fe_df = attach_calendar_targets(fe_df, calendar_data, return_col="simple_return")

# Hapus baris (indikator/target belom ada)
clean_fe_df = fe_df.dropna(subset=features + [target]).copy().reset_index(drop=True)

# Cek tanggal target selalu setelah tanggal fitur
assert (clean_fe_df["target_date"] > clean_fe_df["snapshot_date"]).all()

In [ ]:
# Jumlah Baris Sebelum dan Sesudah Feature Engineering
feature_summary = pd.DataFrame(
    {
        "Keterangan": [
            "Baris sebelum feature engineering",
            "Baris dengan feature dan target lengkap",
            "Jumlah feature",
        ],
        "Nilai": [len(calendar_data), len(clean_fe_df), len(features)],
    }
)
show_table(feature_summary, display_names=display_names)

Contoh berikut menampilkan fitur harga BBCA beserta tanggal target dan return yang akan diprediksi.

In [ ]:
# Sample Data BBCA: tanggal fitur, tanggal target, harga, dan target return
show_table(
    clean_fe_df.loc[
        clean_fe_df["Kode Saham"] == "BBCA",
        ["snapshot_date", "target_date", "close_adj", target],
    ].head(),
    display_names=display_names,
)

## Grafik Harga dan Return

Adjusted close dan daily return sebelas saham ditampilkan untuk periode 2020 sampai 2025.

In [ ]:
# Grafik Adjusted Close dan Daily Return 11 Saham
blue = "#1F5FA8"
n = len(tickers)

fig, axes = plt.subplots(
    n, 2, figsize=(16, 2.4 * n), facecolor="#F8F9FA", sharex=True
)
fig.suptitle(
    "Adjusted Close dan Daily Return Emiten Perbankan (2020 sampai 2025)",
    fontsize=14,
    fontweight="bold",
    color=blue,
    y=0.995,
)

# Satu baris grafik untuk tiap saham
for row, ticker in enumerate(tickers):
    stock = (
        clean_fe_df[clean_fe_df["Kode Saham"] == ticker]
        .sort_values("snapshot_date")
        .copy()
    )
    # Daily return dari adjusted close (desimal, bukan persen)
    stock["daily_return"] = stock["close_adj"].pct_change()

    # Kolom kiri: adjusted close
    ax_price = axes[row, 0]
    ax_price.plot(stock["snapshot_date"], stock["close_adj"], color=blue, linewidth=1.0)
    ax_price.set_ylabel(ticker, fontsize=9, fontweight="bold", color=blue)
    ax_price.grid(alpha=0.3)

    # Kolom kanan: daily return sebagai batang dari garis nol
    ax_ret = axes[row, 1]
    ax_ret.bar(
        stock["snapshot_date"],
        stock["daily_return"],
        color=blue,
        width=1.0,
        linewidth=0,
    )
    ax_ret.axhline(0, color="black", linewidth=0.6)
    ax_ret.grid(alpha=0.3)

    # Judul kolom cuma di baris pertama
    if row == 0:
        ax_price.set_title("Adjusted Close (Rp)", fontsize=10, fontweight="bold", color=blue)
        ax_ret.set_title("Daily Return", fontsize=10, fontweight="bold", color=blue)

# Label tahun cuma di grafik paling bawah
for ax in axes[-1]:
    ax.set_xlabel("Tanggal", fontsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## Simpan Hasil Preprocessing

Data dengan fitur dan target lengkap disimpan ke panel_features.parquet untuk pemodelan. Tabel berikut merangkum file hasil preprocessing beserta jumlah barisnya.

In [ ]:
# Simpan Data fitur dan target untuk notebook 03
clean_fe_df.to_parquet(processed_dir / "panel_features.parquet", index=False)

# Display file yang disimpan beserta jumlah barisnya
output_summary = pd.DataFrame(
    {
        "File": [
            "panel_features.parquet",
            "panel_calendar.parquet",
        ],
        "Baris": [
            len(clean_fe_df),
            len(calendar_data),
        ],
    }
)
show_table(output_summary)